# 16.01: Retraining Fine-Tuned MiniLM Topic Extractor (v2) on 37.5K Educational Dataset

This notebook performs the additive fine-tuning of `sentence-transformers/all-MiniLM-L6-v2` across all 111 Canonical Math Skills.
It integrates:
1. **Original Seed Dataset** (4,499 samples)
2. **Meta-Questions Dataset** (`student_skill_questions.csv`, 11,000 samples)
3. **Raw Equations/Numbers Dataset** (`student_skill_questions_v2.csv`, 11,000 samples)
4. **Real-World Word Problems Dataset** (`student_skill_questions_realworld.csv`, 11,000 samples)

Total dataset: **37,499 deduplicated samples across all 111 canonical skills**.


In [ ]:
# Cell 1: Environment & Imports Setup
from __future__ import annotations

import json
from pathlib import Path
import random
import re
import sys
import numpy as np
import pandas as pd
from sentence_transformers import InputExample, SentenceTransformer
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "topic_extraction"
RAW_DIR = DATA_DIR / "raw"
SPLITS_DIR = DATA_DIR / "splits_v2"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "topic_extractor"
V2_MODEL_DIR = ARTIFACTS_DIR / "minilm_finetuned_v2"
V2_CENTROIDS_NPZ = ARTIFACTS_DIR / "minilm_finetuned_v2_skill_centroids.npz"
V2_MODEL_DIR.mkdir(parents=True, exist_ok=True)

ONTOLOGY_PATH = PROJECT_ROOT / "data" / "processed" / "canonical_skill_ontology.json"

print(f"Project Root: {PROJECT_ROOT}")
print(f"CUDA Available: {torch.cuda.is_available()}")


In [ ]:
# Cell 2: Ontology Loading & Canonical Mapping Normalization
with open(ONTOLOGY_PATH, "r", encoding="utf-8") as f:
    ontology = json.load(f)

print(f"Total Canonical Skills: {len(ontology)}")

display_to_canon = {}
name_to_canon = {}
skill_rep_map = {}

for skill in ontology:
    c_name = skill["canonical_name"].strip()
    d_name = skill["display_name"].strip()
    cat = skill.get("category", "")
    desc = skill.get("description", "")
    rep_text = f"{d_name} ({cat}): {desc}" if desc else f"{d_name} ({cat})"
    
    display_to_canon[d_name] = c_name
    name_to_canon[d_name.lower().strip()] = c_name
    name_to_canon[c_name.lower().strip()] = c_name
    skill_rep_map[c_name] = {
        "metadata": skill,
        "rep_text": rep_text,
    }

def map_to_canonical(skill_name: str) -> str:
    s = str(skill_name).strip()
    if s in display_to_canon:
        return display_to_canon[s]
    s_clean = s.lower().strip()
    if s_clean in name_to_canon:
        return name_to_canon[s_clean]
    # Handle known slight spelling variants from CSVs
    if "parts of a polynomial" in s_clean or "parts of a polyomial" in s_clean:
        return "math :: algebra & functions :: parts of a polyomial, terms, coefficient, monomial, exponent, variable"
    if "effect of changing dimensions" in s_clean:
        return "math :: geometry & measurement :: effect of changing dimensions of a shape prportionally"
    raise ValueError(f"Unrecognized skill name: '{skill_name}'")

print("Ontology mapped successfully!")


In [ ]:
# Cell 3: Load Original Benchmark Data + 3 New Synthetic Datasets
# 1. Original Dataset
orig_train = pd.read_csv(DATA_DIR / "splits" / "train.csv")
orig_val = pd.read_csv(DATA_DIR / "splits" / "validation.csv")
orig_test = pd.read_csv(DATA_DIR / "splits" / "test.csv")
orig_all = pd.concat([orig_train, orig_val, orig_test], ignore_index=True)
orig_records = []
for _, row in orig_all.iterrows():
    orig_records.append({
        "question": str(row["text"]).strip(),
        "canonical_name": str(row["canonical_name"]).strip(),
        "display_name": str(row["skill_name"]).strip(),
        "source": "original_benchmark_v1",
    })
df_orig = pd.DataFrame(orig_records)

# 2. 3 New CSVs
csv1 = pd.read_csv(RAW_DIR / "student_skill_questions.csv")
csv2 = pd.read_csv(RAW_DIR / "student_skill_questions_v2.csv")
csv3 = pd.read_csv(RAW_DIR / "student_skill_questions_realworld.csv")

def parse_new_csv(df: pd.DataFrame, source_tag: str) -> pd.DataFrame:
    records = []
    for _, row in df.iterrows():
        c_name = map_to_canonical(row["skill_name"])
        records.append({
            "question": str(row["question"]).strip(),
            "canonical_name": c_name,
            "display_name": str(row["skill_name"]).strip(),
            "source": source_tag,
        })
    return pd.DataFrame(records)

df_new1 = parse_new_csv(csv1, "synthetic_meta_questions")
df_new2 = parse_new_csv(csv2, "synthetic_equations_v2")
df_new3 = parse_new_csv(csv3, "synthetic_realworld_story")

merged_df = pd.concat([df_orig, df_new1, df_new2, df_new3], ignore_index=True)
merged_df = merged_df.drop_duplicates(subset=["question"]).reset_index(drop=True)

print(f"Original v1 Rows:         {len(df_orig)}")
print(f"New CSV 1 (Meta):         {len(df_new1)}")
print(f"New CSV 2 (Equations):    {len(df_new2)}")
print(f"New CSV 3 (Real-World):   {len(df_new3)}")
print(f"Total Merged & Deduped:   {len(merged_df)} samples across {merged_df['canonical_name'].nunique()} Canonical Skills")


In [ ]:
# Cell 4: Stratified 80/10/10 Split with Full Global & Class Shuffling
random.seed(42)
np.random.seed(42)

train_rows, val_rows, test_rows = [], [], []

for c_name, group in merged_df.groupby("canonical_name"):
    tr, temp = train_test_split(group, test_size=0.20, random_state=42, shuffle=True)
    va, te = train_test_split(temp, test_size=0.50, random_state=42, shuffle=True)
    train_rows.append(tr)
    val_rows.append(va)
    test_rows.append(te)

# Fully shuffle combined splits to eliminate any alphabetical ordering bias
train_df = pd.concat(train_rows, ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)
val_df = pd.concat(val_rows, ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)
test_df = pd.concat(test_rows, ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)

test_realworld_slice = test_df[test_df["source"] == "synthetic_realworld_story"].reset_index(drop=True)

train_df.to_csv(SPLITS_DIR / "train.csv", index=False)
val_df.to_csv(SPLITS_DIR / "validation.csv", index=False)
test_df.to_csv(SPLITS_DIR / "test.csv", index=False)
test_realworld_slice.to_csv(SPLITS_DIR / "test_realworld_slice.csv", index=False)

print(f"Train Set:               {len(train_df)} samples ({train_df['canonical_name'].nunique()} skills)")
print(f"Validation Set:          {len(val_df)} samples ({val_df['canonical_name'].nunique()} skills)")
print(f"Test Set (Full Held-out):{len(test_df)} samples ({test_df['canonical_name'].nunique()} skills)")
print(f"Real-World Test Slice:   {len(test_realworld_slice)} samples")


In [ ]:
# Cell 5: Fine-Tuning SentenceTransformer (all-MiniLM-L6-v2) on Merged Dataset
train_examples = []
for _, row in train_df.iterrows():
    c_name = row["canonical_name"]
    target_text = skill_rep_map[c_name]["rep_text"]
    train_examples.append(InputExample(texts=[str(row["question"]), str(target_text)]))

print(f"Prepared {len(train_examples)} contrastive training pairs across 111 skills.")
torch.set_num_threads(10)

finetuned_v1_path = ARTIFACTS_DIR / "minilm_finetuned"
if finetuned_v1_path.exists():
    print(f"Loading existing fine-tuned model checkpoint for additive learning: {finetuned_v1_path}")
    model = SentenceTransformer(str(finetuned_v1_path), device="cpu")
else:
    print("Loading base model: sentence-transformers/all-MiniLM-L6-v2")
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")

model.max_seq_length = 64

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=128)
train_loss = MultipleNegativesRankingLoss(model)

print("Starting Fine-Tuning (1 Epoch on 30k merged pairs, batch_size=128, lr=2e-5)...")
warmup_steps = int(len(train_dataloader) * 1 * 0.1)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=1,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": 2e-5},
    show_progress_bar=True,
)
print("Fine-tuning completed successfully!")


In [ ]:
# Cell 6: Compute Skill Centroids & Save v2 Artifacts
def compute_centroids(mdl: SentenceTransformer, data_df: pd.DataFrame, ont: list[dict]):
    canonical_names = [s["canonical_name"] for s in ont]
    name_to_idx = {name: i for i, name in enumerate(canonical_names)}
    texts = data_df["question"].astype(str).tolist()
    embeddings = mdl.encode(texts, batch_size=128, show_progress_bar=False, normalize_embeddings=True)
    n_skills = len(ont)
    dim = embeddings.shape[1]
    raw_centroids = np.zeros((n_skills, dim), dtype=np.float32)
    for skill_name, group in data_df.groupby("canonical_name"):
        if skill_name in name_to_idx:
            idx = name_to_idx[skill_name]
            skill_vecs = embeddings[group.index.tolist()]
            raw_centroids[idx] = np.mean(skill_vecs, axis=0)
    norm_centroids = normalize(raw_centroids, norm="l2", axis=1)
    return norm_centroids, canonical_names

v2_centroids, canonical_names = compute_centroids(model, train_df, ontology)

model.save(str(V2_MODEL_DIR))
np.savez_compressed(
    V2_CENTROIDS_NPZ,
    centroids=v2_centroids,
    canonical_names=np.array(canonical_names, dtype=object),
)

print(f"Saved Fine-Tuned Model v2 to: {V2_MODEL_DIR}")
print(f"Saved Skill Centroids v2 to:    {V2_CENTROIDS_NPZ}")


In [ ]:
# Cell 7: Full Held-Out Test Evaluation & Benchmark Comparison
def evaluate_on_set(mdl: SentenceTransformer, centroids: np.ndarray, canonical_names: list[str], eval_df: pd.DataFrame):
    texts = eval_df["question"].astype(str).tolist()
    embeddings = mdl.encode(texts, batch_size=128, show_progress_bar=False, normalize_embeddings=True)
    sims_matrix = np.dot(embeddings, centroids.T)
    
    top1_indices = np.argmax(sims_matrix, axis=1)
    top3_indices = np.argsort(sims_matrix, axis=1)[:, -3:]
    
    y_true = eval_df["canonical_name"].tolist()
    y_pred = [canonical_names[idx] for idx in top1_indices]
    
    acc = accuracy_score(y_true, y_pred)
    
    top3_correct = 0
    for row_idx, true_name in enumerate(y_true):
        cand_names = [canonical_names[idx] for idx in top3_indices[row_idx]]
        if true_name in cand_names:
            top3_correct += 1
    top3_acc = top3_correct / len(eval_df)
    
    all_classes = sorted(canonical_names)
    macro_f1 = f1_score(y_true, y_pred, labels=all_classes, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, labels=all_classes, average="weighted", zero_division=0)
    
    return {
        "accuracy": float(acc),
        "top3_accuracy": float(top3_acc),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "y_true": y_true,
        "y_pred": y_pred,
        "sims_matrix": sims_matrix,
    }

test_results = evaluate_on_set(model, v2_centroids, canonical_names, test_df)
realworld_results = evaluate_on_set(model, v2_centroids, canonical_names, test_realworld_slice)

print("=" * 80)
print("BENCHMARK COMPARISON TABLE")
print("=" * 80)
print(f"{'Model':<42} | {'Accuracy':<10} | {'Top-3 Acc':<10} | {'Macro F1':<10}")
print("-" * 80)
print(f"{'1. Keyword / Alias Matcher':<42} | {'87.83%':<10} | {'N/A':<10} | {'0.8948':<10}")
print(f"{'2. TF-IDF + Cosine Centroids':<42} | {'95.94%':<10} | {'97.97%':<10} | {'0.9654':<10}")
print(f"{'3. Pretrained MiniLM (Base)':<42} | {'95.01%':<10} | {'98.91%':<10} | {'0.9487':<10}")
print(f"{'4. Fine-Tuned MiniLM v1 (Original 4.5K)':<42} | {'98.91%':<10} | {'99.69%':<10} | {'0.9913':<10}")
print(f"{'5. Fine-Tuned MiniLM v2 (Merged 37.5K)':<42} | {test_results['accuracy']*100:.2f}%     | {test_results['top3_accuracy']*100:.2f}%     | {test_results['macro_f1']:.4f}")
print("=" * 80)


In [ ]:
# Cell 8: Real-World Story Word Problems Evaluation & User Example Probe
print(f"Real-World Story Problem Test Accuracy: {realworld_results['accuracy']*100:.2f}% (Top-3: {realworld_results['top3_accuracy']*100:.2f}%)")

# Test user query:
train_q = "A train leaves Colombo at 8:35 AM and reaches its destination at 11:20 AM. How long was the journey?"
q_emb = model.encode([train_q], normalize_embeddings=True)
q_sims = np.dot(q_emb, v2_centroids.T)[0]
top_idx = np.argmax(q_sims)
top_score = float(q_sims[top_idx])
top_disp_name = [s["display_name"] for s in ontology if s["canonical_name"] == canonical_names[top_idx]][0]

print(f"\nProbe on User Story Question: '{train_q}'")
print(f" -> Predicted Topic:   {top_disp_name}")
print(f" -> Top-1 Similarity:  {top_score:.4f}")


In [ ]:
# Cell 9: Regression Check Against Original v1 Test Set
orig_test_df = pd.read_csv(DATA_DIR / "splits" / "test.csv")
orig_test_df["question"] = orig_test_df["text"]
orig_test_eval = evaluate_on_set(model, v2_centroids, canonical_names, orig_test_df)

print(f"Original v1 Test Accuracy on v2 Model: {orig_test_eval['accuracy']*100:.2f}% (Top-3: {orig_test_eval['top3_accuracy']*100:.2f}%, Macro F1: {orig_test_eval['macro_f1']:.4f})")


In [ ]:
# Cell 10: Confusion Matrix & Top Confused Skill Pairs
y_true_all = test_results["y_true"]
y_pred_all = test_results["y_pred"]

confused_dict = {}
for yt, yp in zip(y_true_all, y_pred_all):
    if yt != yp:
        pair = (yt, yp)
        confused_dict[pair] = confused_dict.get(pair, 0) + 1

sorted_confused = sorted(confused_dict.items(), key=lambda x: x[1], reverse=True)
print(f"Total Confused Pairs: {len(sorted_confused)}")
print("\nTop 10 Most Confused Skill Pairs:")
for idx, ((yt, yp), cnt) in enumerate(sorted_confused[:10], 1):
    yt_disp = [s["display_name"] for s in ontology if s["canonical_name"] == yt][0]
    yp_disp = [s["display_name"] for s in ontology if s["canonical_name"] == yp][0]
    print(f"  {idx:2d}. True: [{yt_disp:<35}] -> Predicted: [{yp_disp:<35}] (Count: {cnt})")


In [ ]:
# Cell 11: Calibrate Confidence & Safe Abstention Threshold
math_sims = np.max(test_results["sims_matrix"], axis=1)

OOD_TEST_QUERIES = [
    "Hello there, how are you doing today?",
    "Hi, who are you?",
    "Good morning tutor",
    "What is your name?",
    "Tell me a funny joke or story",
    "How was your day?",
    "Can you write a poem about the autumn leaves?",
    "What is the process of photosynthesis in green plants?",
    "What were the main causes of World War 1?",
    "Who was the first president of the United States?",
    "What is the capital city of Australia?",
    "How do I create a React web application with Vite?",
    "What is the theme of The Great Gatsby?",
    "I am feeling bored right now",
    "Help me please",
    "Can you tell me the answer right now?",
]

ood_emb = model.encode(OOD_TEST_QUERIES, show_progress_bar=False, normalize_embeddings=True)
ood_sims = np.max(np.dot(ood_emb, v2_centroids.T), axis=1)

print(f"Math Test Max Similarity:  Mean={np.mean(math_sims):.4f}, 5th-Percentile={np.percentile(math_sims, 5):.4f}")
print(f"Off-Topic Max Similarity: Mean={np.mean(ood_sims):.4f}, Max={np.max(ood_sims):.4f}")

rec_threshold = 0.25
ood_rejection = np.mean(ood_sims < rec_threshold)
math_acceptance = np.mean(math_sims >= rec_threshold)

print(f"\nRecommended Threshold: {rec_threshold:.2f}")
print(f"OOD Rejection Rate:    {ood_rejection * 100:.1f}%")
print(f"Math Acceptance Rate:  {math_acceptance * 100:.1f}%")


In [ ]:
# Cell 12: Save Comprehensive Test Metrics & Update model_selection.json
prec, rec, f1, supp = precision_recall_fscore_support(
    y_true_all,
    y_pred_all,
    labels=sorted(canonical_names),
    zero_division=0,
)

per_skill_metrics = []
for c_name, p, r, f, s in zip(sorted(canonical_names), prec, rec, f1, supp):
    d_name = [item["display_name"] for item in ontology if item["canonical_name"] == c_name][0]
    per_skill_metrics.append({
        "canonical_name": c_name,
        "display_name": d_name,
        "precision": float(p),
        "recall": float(r),
        "f1": float(f),
        "test_support": int(s),
    })

metrics_summary = {
    "model_name": "Fine-Tuned MiniLM v2 (Merged 37.5K)",
    "base_architecture": "sentence-transformers/all-MiniLM-L6-v2",
    "loss_function": "MultipleNegativesRankingLoss",
    "total_training_samples": len(train_df),
    "total_validation_samples": len(val_df),
    "total_test_samples": len(test_df),
    "benchmark": {
        "test_accuracy": test_results["accuracy"],
        "test_top3_accuracy": test_results["top3_accuracy"],
        "test_macro_f1": test_results["macro_f1"],
        "test_weighted_f1": test_results["weighted_f1"],
        "realworld_slice_accuracy": realworld_results["accuracy"],
        "realworld_slice_top3_accuracy": realworld_results["top3_accuracy"],
        "v1_regression_test_accuracy": orig_test_eval["accuracy"],
    },
    "recommended_threshold": rec_threshold,
    "top_confused_pairs": sorted_confused[:15],
    "per_skill_metrics": per_skill_metrics,
}

with open(ARTIFACTS_DIR / "final_test_metrics_v2.json", "w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, indent=2)

# Update model_selection.json with comparative entries, marking v1 as currently active in production
model_sel_path = ARTIFACTS_DIR / "model_selection.json"
model_sel_data = {
    "active_production_model": "minilm_finetuned",
    "candidate_upgrade_model": "minilm_finetuned_v2",
    "rule_model": "keyword_alias",
    "primary_semantic_model": "minilm_finetuned",
    "fallback_model": "tfidf",
    "selection_metric": "validation_macro_f1",
    "benchmarks": [
        {
            "model_name": "Keyword / Rule Baseline",
            "accuracy": 0.8783,
            "macro_f1": 0.8948,
            "weighted_f1": None,
            "top3_accuracy": None,
            "status": "baseline",
        },
        {
            "model_name": "TF-IDF + Cosine Centroids",
            "accuracy": 0.9594,
            "macro_f1": 0.9654,
            "weighted_f1": 0.9585,
            "top3_accuracy": 0.9797,
            "status": "fallback",
        },
        {
            "model_name": "Pretrained MiniLM (all-MiniLM-L6-v2)",
            "accuracy": 0.9501,
            "macro_f1": 0.9487,
            "weighted_f1": 0.9500,
            "top3_accuracy": 0.9891,
            "status": "base",
        },
        {
            "model_name": "Fine-Tuned MiniLM v1 (Domain Adapted 4.5K)",
            "accuracy": 0.9891,
            "macro_f1": 0.9913,
            "weighted_f1": 0.9892,
            "top3_accuracy": 0.9969,
            "status": "active_production",
        },
        {
            "model_name": "Fine-Tuned MiniLM v2 (Merged 37.5K Extended)",
            "accuracy": float(test_results["accuracy"]),
            "macro_f1": float(test_results["macro_f1"]),
            "weighted_f1": float(test_results["weighted_f1"]),
            "top3_accuracy": float(test_results["top3_accuracy"]),
            "realworld_accuracy": float(realworld_results["accuracy"]),
            "v1_regression_accuracy": float(orig_test_eval["accuracy"]),
            "status": "ready_for_review",
        }
    ]
}

with open(model_sel_path, "w", encoding="utf-8") as f:
    json.dump(model_sel_data, f, indent=2)

print("Saved metrics and updated model_selection.json successfully!")
